# RG3 가스: 절대 운전 상태 + 흔들림

관계 잔차에서 원래 수준 정보가 사라지는 가설을 확인. Hopper/Barrel2/Barrel5/쿠션/계량시간/최대압력-전환압력 차이 6개 + 과거10개 흔들림4 = 10개. std4 기준, state10, 전문가 최대 점수 앙상블. RF/ET 두 시드 평균. 임계값 선택은 내부 구간에서만.

RG3 RH 정상601/가스19, CN7 미사용. 날짜/Run 번호/정답 이웃값을 입력하지 않음. 7개 관측구간 중첩검증은 새 독립 검증 아님. 실험 결과를 보고 추가한 개발 가설이며 테스트 일반화 주장 금지.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import ExtraTreesRegressor,RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
assert len(d)==620 and y.sum()==19
inputs=['Max_Injection_Speed','Average_Screw_RPM','Barrel_Temperature_2','Barrel_Temperature_5','Plasticizing_Position','Filling_Time']
targets=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Cushion_Position','Plasticizing_Time']
stdcols=[x+'__past10_std' for x in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
memo={};audit=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 normal=ref[y[ref]==0]
 state=d[['Hopper_Temperature','Barrel_Temperature_2','Barrel_Temperature_5','Cushion_Position','Plasticizing_Time']].copy()
 state['pressure_switch_gap']=d.Max_Injection_Pressure-d.Max_Switch_Over_Pressure
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(state.iloc[normal])
 tr=im.transform(state.iloc[ref]);te=im.transform(state.iloc[q])
 si=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[normal][stdcols])
 st=si.transform(d.iloc[ref][stdcols]);sq=si.transform(d.iloc[q][stdcols])
 features=[(st,sq),(np.c_[tr,st],np.c_[te,sq])]
 scores=[]
 for ft,fq in features:
  vals=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    if len(np.unique(y[ref]))<2:vals.append(np.zeros(len(q)));continue
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
    vals.append(m.fit(ft,y[ref]).predict_proba(fq)[:,1])
  scores.append(np.mean(vals,axis=0))
 memo[key]=np.column_stack([scores[0],scores[1],np.maximum(*scores)])
 audit.append(dict(train=len(ref),query=len(q),normal_imputer_training=len(normal),gas_training=int(y[ref].sum()),train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 out=predict(ref,q);inner=np.full((len(ref),3),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['std4_reference','absolute_state10','std4_state_max_ensemble']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   pp=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],pp,labels=[0,1]).ravel()
   rec=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((rec,f1,-fp,name,budget,out[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(out[j,k]>cut),score=float(out[j,k]),threshold=cut))
 chosen=max(candidates,key=lambda x:x[:3])
 choices.append(dict(block=int(b),selected=chosen[3],budget=chosen[4],inner_recall=chosen[0],inner_F1=chosen[1]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(chosen[5][j]),score=np.nan,threshold=np.nan))
 print('Finished RG3 conditional relationship block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget),g in p.groupby(['model','budget']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(g.actual,g.score) if g.score.notna().all() else np.nan))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(tables/f'rg3_absolute_state_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))



Finished RG3 conditional relationship block 1


Finished RG3 conditional relationship block 2


Finished RG3 conditional relationship block 3


Finished RG3 conditional relationship block 4


Finished RG3 conditional relationship block 5


Finished RG3 conditional relationship block 6


Finished RG3 conditional relationship block 7


                      model  budget  TP  FP  FN  TN   recall       F1       AP
           absolute_state10    0.02   3  23  16 578 0.157895 0.133333 0.063661
           absolute_state10    0.05   3  45  16 556 0.157895 0.089552 0.063661
           absolute_state10    0.10   4  86  15 515 0.210526 0.073394 0.063661
inner_recall_first_selected    0.00   3  66  16 535 0.157895 0.068182      NaN
             std4_reference    0.02   3  26  16 575 0.157895 0.125000 0.061578
             std4_reference    0.05   3  33  16 568 0.157895 0.109091 0.061578
             std4_reference    0.10   3  67  16 534 0.157895 0.067416 0.061578
    std4_state_max_ensemble    0.02   3  26  16 575 0.157895 0.125000 0.059705
    std4_state_max_ensemble    0.05   3  33  16 568 0.157895 0.109091 0.059705
    std4_state_max_ensemble    0.10   3  65  16 536 0.157895 0.068966 0.059705
